# Hierarchical Equality — task demo

Three sections:

1. **Causal model** — the four-letter `(a==b) == (c==d)` DAG.
2. **Templates & token positions** — prompt rendering across the three modes, plus how the test-query variable positions are located inside a 60-example ICL prompt.
3. **Counterfactual generators** — what the balanced random sampler produces.

Tokenization uses `gpt2`. No interventions — see `analyses/locate/demo.ipynb` for those.

In [1]:
from causalab.tasks.hierarchical_equality.causal_models import CAUSAL_MODEL
from causalab.tasks.hierarchical_equality.counterfactuals import (
    sample_balanced_input,
    generate_dataset,
)
from causalab.tasks.hierarchical_equality.token_positions import create_token_positions
from causalab.tasks.hierarchical_equality.config import (
    PATTERNS,
    PROMPT_MODE,
    NUM_ICL_EXAMPLES,
)

PROMPT_MODE, NUM_ICL_EXAMPLES, PATTERNS

('minimal_function', 60, ['AABB', 'ABCD', 'ABCC', 'AABC'])

## 1. Causal model

Inputs: `var_1`, `var_2`, `var_3`, `var_4`, `template`. Computed: `left_equality`, `right_equality`, `result_equality`, `raw_input`, `raw_output`.

Sample one example per pattern (`AABB`, `ABCD`, `ABCC`, `AABC`) and inspect the trace. The `raw_input` is huge (60 ICL examples) so we'll truncate the display.

In [2]:
import random
from causalab.tasks.hierarchical_equality.templates import (
    _sample_pattern_values,
    TEMPLATES,
)  # pyright: ignore[reportPrivateUsage]

random.seed(0)
for pattern in PATTERNS:
    v1, v2, v3, v4 = _sample_pattern_values(pattern)
    trace = CAUSAL_MODEL.new_trace(
        {
            "template": TEMPLATES[0],
            "var_1": v1,
            "var_2": v2,
            "var_3": v3,
            "var_4": v4,
        }
    )
    print(f"--- pattern {pattern} ---")
    print(
        f"  vars=({v1}, {v2}, {v3}, {v4}) -> "
        f"left={trace['left_equality']}, right={trace['right_equality']}, "
        f"result={trace['result_equality']} -> raw_output={trace['raw_output']!r}"
    )

--- pattern AABB ---
  vars=(M, M, Y, Y) -> left=True, right=True, result=True -> raw_output='1'
--- pattern ABCD ---
  vars=(W, J, Y, O) -> left=False, right=False, result=True -> raw_output='1'
--- pattern ABCC ---
  vars=(U, E, T, T) -> left=False, right=True, result=False -> raw_output='0'
--- pattern AABC ---
  vars=(X, X, X, M) -> left=True, right=False, result=False -> raw_output='0'


Two patterns produce `1` (when both equalities match: AABB → T==T, ABCD → F==F) and two produce `0` (ABCC → F==T, AABC → T==F). The four patterns are sampled uniformly by `sample_balanced_input`, so the dataset is balanced over `(left_equality, right_equality)` pairs by construction.

In [3]:
trace = sample_balanced_input()
lines = trace["raw_input"].split("\n")
print(
    f"Mode: {PROMPT_MODE}    ICL examples: {len(lines) - 1}    expected output: {trace['raw_output']!r}"
)
print()
print("First 3 ICL lines:")
for line in lines[:3]:
    print(f"  {line!r}")
print("...")
print("Last 3 lines (test query is the very last):")
for line in lines[-3:]:
    print(f"  {line!r}")

Mode: minimal_function    ICL examples: 60    expected output: '0'

First 3 ICL lines:
  'f(P,B,R,Q)=1'
  'f(V,V,V,V)=1'
  'f(R,B,Y,I)=1'
...
Last 3 lines (test query is the very last):
  'f(Q,J,Z,I)=1'
  'f(J,X,I,I)=0'
  'f(F,G,X,X)='


## 2. Templates & token positions

ICL prompts repeat letters many times, so the test-query variables can't be located by "last occurrence". `create_token_positions` parses the final line with a `PROMPT_MODE`-specific regex (here `(?<=[(,])([^,)]+)` for `minimal_function`) and maps the character span to token indices.

In [4]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1, max_length=2048)
positions = create_token_positions(pipeline)
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['last', 'var_1', 'var_2', 'var_3', 'var_4']

In [5]:
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id

# Only show the tail of the prompt (test query plus a few ICL lines for context)
non_pad_indices = [i for i, t in enumerate(ids) if t != pad_id]
tail_start = max(0, len(non_pad_indices) - 25)
tail = non_pad_indices[tail_start:]

print("Test query (last line of raw_input):", trace["raw_input"].splitlines()[-1])
print()
print(f"{'idx':>5}  {'token':<14}  positions")
for i in tail:
    hits = [name for name, pos in positions.items() if i in pos.index(trace)]
    marker = ", ".join(hits) if hits else ""
    print(f"{i:>5}  {decoded[i]!r:<14}  {marker}")

Test query (last line of raw_input): f(F,G,X,X)=

  idx  token           positions
 2023  ')='            
 2024  '1'             
 2025  '\n'            
 2026  'f'             
 2027  '('             
 2028  'J'             
 2029  ','             
 2030  'X'             
 2031  ','             
 2032  'I'             
 2033  ','             
 2034  'I'             
 2035  ')='            
 2036  '0'             


 2037  '\n'            
 2038  'f'             
 2039  '('             
 2040  'F'             var_1
 2041  ','             
 2042  'G'             var_2
 2043  ','             
 2044  'X'             var_3
 2045  ','             
 2046  'X'             var_4
 2047  ')='            last


## 3. Counterfactual generators

`generate_dataset(model, n, seed)` returns `n` `{input, counterfactual_inputs}` pairs. Both base and counterfactual are independent calls to `sample_balanced_input` (uniform over `PATTERNS`), so every `(left_equality, right_equality)` combination is equally represented.

In [6]:
for i, ex in enumerate(generate_dataset(CAUSAL_MODEL, n=4, seed=0)):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- pair {i} ---")
    print(
        f"  base : vars=({base['var_1']}, {base['var_2']}, {base['var_3']}, {base['var_4']})  "
        f"-> left={base['left_equality']}, right={base['right_equality']}, result={base['result_equality']}"
    )
    print(
        f"  cf   : vars=({cf['var_1']}, {cf['var_2']}, {cf['var_3']}, {cf['var_4']})  "
        f"-> left={cf['left_equality']}, right={cf['right_equality']}, result={cf['result_equality']}"
    )
    print()

--- pair 0 ---
  base : vars=(Y, Y, N, B)  -> left=True, right=False, result=False
  cf   : vars=(Y, O, Q, Q)  -> left=False, right=True, result=False

--- pair 1 ---
  base : vars=(T, Q, C, I)  -> left=False, right=False, result=True
  cf   : vars=(R, R, E, Z)  -> left=True, right=False, result=False

--- pair 2 ---
  base : vars=(Y, H, D, D)  -> left=False, right=True, result=False
  cf   : vars=(T, E, Q, Z)  -> left=False, right=False, result=True

--- pair 3 ---
  base : vars=(W, X, T, T)  -> left=False, right=True, result=False
  cf   : vars=(O, P, Z, F)  -> left=False, right=False, result=True



## Next steps

Run the locate / subspace pipeline:

```bash
./scripts/run_exp.sh he_locate
./scripts/run_exp.sh he_subspace
./scripts/run_exp.sh he_pipeline
```

Outputs land under `artifacts/hierarchical_equality/<model>/<analysis>/...`.